In [9]:
# Unified pipeline: from a single user input -> structured JSON -> top-5 cost matches
from pathlib import Path
import sys, os, json
from dotenv import load_dotenv

# Ensure repo root on path
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load env (OPENAI_API_KEY)
load_dotenv(dotenv_path=repo_root/".env")

# 1) Text parsing and structuring
from scripts.text_input import generate_final_output

# 2) Keyword maps (avoid importing full cost_estimation to keep things light)
from scripts.config.material_keywords import MATERIAL_TYPE_KEYWORDS
from scripts.config.element_keywords import ELEMENT_TYPE_KEYWORDS

global_keywords_map = {**MATERIAL_TYPE_KEYWORDS, **ELEMENT_TYPE_KEYWORDS}

# 3) Helpers mirrored from cost_estimation
import re

def preprocess_text_with_keywords(text, keyword_map):
    if not isinstance(text, str):
        return ''
    processed_text = text.lower()
    added_keywords = set()
    for main_term, keywords_list in keyword_map.items():
        if main_term.lower() in processed_text:
            for keyword in keywords_list:
                added_keywords.add(keyword.lower())
    parts = [text]
    for kw in added_keywords:
        if kw not in processed_text:
            parts.append(kw)
    return ' '.join(parts)


def extract_text_from_json(json_obj):
    text_parts = []
    # element_type
    if 'element_type' in json_obj and 'value' in json_obj['element_type'] and json_obj['element_type']['value'] is not None:
        text_parts.append(json_obj['element_type']['value'])
    # materials.primary
    if 'materials' in json_obj and json_obj['materials'] and 'primary' in json_obj['materials']:
        primary_material = json_obj['materials']['primary'] or {}
        if 'material_type' in primary_material and primary_material['material_type'] is not None:
            text_parts.append(primary_material['material_type'])
        if 'specification' in primary_material and primary_material['specification'] is not None:
            text_parts.append(primary_material['specification'])
        if 'keywords' in primary_material and primary_material['keywords'] is not None:
            text_parts.extend(primary_material['keywords'])
    # construction_category
    if 'construction_category' in json_obj and json_obj['construction_category'] and 'value' in json_obj['construction_category'] and json_obj['construction_category']['value'] is not None:
        text_parts.append(json_obj['construction_category']['value'])
    return ' '.join(filter(None, text_parts)).strip()

# 4) Load dataset and build DF text
import pandas as pd

df = pd.read_excel(repo_root/"database"/"Data AIRE project.xlsx")
df['combined_text_df'] = (df['Domain'].fillna('') + ' ' +
                          df['Subdomain'].fillna('') + ' ' +
                          df['product_material'].fillna('') + ' ' +
                          df['related_terms_pt'].fillna('') + ' ' +
                          df['related_terms_en'].fillna('') + ' ' +
                          df['technical_specs'].fillna(''))

df['combined_text_df'] = df['combined_text_df'].apply(lambda x: preprocess_text_with_keywords(x, global_keywords_map))

# 5) Embeddings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
df_embeddings = model.encode(df['combined_text_df'].tolist())

# 6) Single user input -> structured -> combined text -> top-5 matches
USER_INPUT = "Steel beam S275 with length of 6 m"  # change here
structured = generate_final_output(USER_INPUT)
combined_text = preprocess_text_with_keywords(extract_text_from_json(structured), global_keywords_map)

user_vec = model.encode(combined_text).reshape(1, -1)
sims = cosine_similarity(user_vec, df_embeddings)[0]
idxs = np.argsort(sims)[-5:][::-1]

results_summary = []
print("Top 5 matches:")
for j, idx in enumerate(idxs):
    row = df.iloc[idx]
    sim = float(sims[idx])
    unit_prices_val = float(row['unit_prices']) if pd.notna(row['unit_prices']) else None
    measurement_units_val = str(row['measurement_units']) if pd.notna(row['measurement_units']) else None
    price_display = "N/A"
    if unit_prices_val is not None and measurement_units_val is not None:
        price_display = f"{unit_prices_val}€/{measurement_units_val}"
    elif unit_prices_val is not None:
        price_display = f"{unit_prices_val}€"
    elif measurement_units_val is not None:
        price_display = f"N/A / {measurement_units_val}"
    print(f"  {j+1}. Similarity: {sim:.4f} - {row['product_material']} ({row['Domain']} - {row['Subdomain']}) - Price: {price_display if j==0 else 'N/A'}")
    results_summary.append({
        "rank": j+1,
        "similarity": sim,
        "product_material": str(row['product_material']),
        "domain": str(row['Domain']),
        "subdomain": str(row['Subdomain']),
        "unit_prices": unit_prices_val,
        "measurement_units": measurement_units_val,
    })

#print("\nStructured output:")
#print(json.dumps(structured, indent=2, ensure_ascii=False))
#print("\nresults_summary (dict list) is available in the notebook scope.")


Top 5 matches:
  1. Similarity: 0.8163 - steel beams IPN, IPE, HEB, HEA, HEM or UPN  (Steel Structures - Beams) - Price: 1.54€/kg
  2. Similarity: 0.7662 - A400 NR Factory fabricated reinforcement  (Concrete structures - Beams) - Price: N/A
  3. Similarity: 0.7399 - A400 NR Reinforcement Rebars (Concrete structures - Beams) - Price: N/A
  4. Similarity: 0.7074 - Tie wire (Concrete structures - Beams) - Price: N/A
  5. Similarity: 0.6960 - steel columns IPN, IPE, HEB, HEA, HEM or UPN  (Steel Structures - Columns) - Price: N/A


In [8]:
# Build generated inputs for cost_estimation from TEST_INPUTS
from pathlib import Path
import sys, json

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.text_input import generate_final_output
from tests.test_inputs import TEST_INPUTS

outputs = []
for i, inp in enumerate(TEST_INPUTS, 1):
    res = generate_final_output(inp)
    outputs.append(res)
    print(f"\n--- Example {i} ---\nInput Text: {inp}")
    print(json.dumps(res, indent=2, ensure_ascii=False))

# Write to tests/generated_inputs.py so cost_estimation.py can import
py_path = Path.cwd() / "generated_inputs.py"  # tests/generated_inputs.py
payload = "GENERATED_INPUTS = [\n" + \
    ",\n".join(json.dumps(o, ensure_ascii=False) for o in outputs) + "\n]\n"
py_path.write_text(payload, encoding="utf-8")
print(f"Written {len(outputs)} entries to {py_path}")



--- Example 1 ---
Input Text: Steel beam S275 with length of 6 m
{
  "element_type": {
    "value": "beam",
    "confidence": 1.0
  },
  "materials": {
    "primary": {
      "material_type": "steel",
      "specification": "S275",
      "keywords": [
        "steel",
        "S275"
      ],
      "matched_db_entry": "Steel",
      "match_confidence": 1.0
    }
  },
  "dimensions": {
    "height": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "width": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "thickness": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "length": {
      "value": 6.0,
      "unit": "m",
      "standardized_value": 6.0,
      "standardized_unit": "m"
    },
    "weight": {
      "value": null,
      "unit": null,
      "standardized_value": null,
   

KeyboardInterrupt: 

In [5]:
from pathlib import Path
import sys, os, json
from dotenv import load_dotenv

# Ensure project root is importable (notebook lives in tests/)
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load environment variables (expects OPENAI_API_KEY in .env at repo root)
load_dotenv(dotenv_path=repo_root/".env")

from scripts.text_input import generate_final_output
from tests.test_inputs import TEST_INPUTS

for i, inp in enumerate(TEST_INPUTS, 1):
    print(f"\n--- Example {i} ---")
    print(f"Input Text: {inp}")
    print(json.dumps(generate_final_output(inp), indent=2, ensure_ascii=False))


--- Example 1 ---
Input Text: Steel beam S275 with length of 6 m
{
  "element_type": {
    "value": "beam",
    "confidence": 1.0
  },
  "materials": {
    "primary": {
      "material_type": "steel",
      "specification": "S275",
      "keywords": [
        "steel",
        "S275"
      ],
      "matched_db_entry": "Steel",
      "match_confidence": 1.0
    }
  },
  "dimensions": {
    "height": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "width": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "thickness": {
      "value": null,
      "unit": null,
      "standardized_value": null,
      "standardized_unit": null
    },
    "length": {
      "value": 6.0,
      "unit": "m",
      "standardized_value": 6.0,
      "standardized_unit": "m"
    },
    "weight": {
      "value": null,
      "unit": null,
      "standardized_value": null,
   